# Multi-Model Excess Return CSV Builder

**Objective:** For each stock (GME, AMC) and each bar frequency (1-min, 5-min, 10-min),
construct a unified CSV containing:
- Raw price and bar return
- SPY return and simple market-adjusted excess return
- Fama-French factors (MKT_RF, SMB, HML, MOM)
- Abnormal returns and cumulative abnormal returns under three models:
  - **CAPM**: `R = α + β·SPY + ε`
  - **FF3**: `R = α + β_MKT·MKT_RF + β_SMB·SMB + β_HML·HML + ε`
  - **FF4**: `R = α + β_MKT·MKT_RF + β_SMB·SMB + β_HML·HML + β_MOM·MOM + ε`

Models are estimated over a **120-day pre-event estimation window** (ending Jan 26, 2021)
**at each bar frequency** (betas estimated on 5-min bars for 5-min residuals, etc.).

**Output:** 6 CSV files — `{SYMBOL}-{freq}-multi-model-excess-return.csv`

---
## 1. Setup

In [ ]:
import pathlib
import warnings
from datetime import timedelta

import numpy as np
import pandas as pd
import statsmodels.api as sm

warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────────────
PROJECT_ROOT = pathlib.Path.cwd().parent
DATA_DIR     = PROJECT_ROOT / "Minute price and exceed return" / "output_data"
FF_DIR       = PROJECT_ROOT / "FF-Factors"
OUTPUT_DIR   = DATA_DIR  # save alongside existing CSVs

# ── Stock definitions ──────────────────────────────────────────────────────────
STOCKS = {
    "GME": {
        "excess_return_file": "GME-minute_price-excess-return.csv",
        "price_col":  "gme_price",
        "return_col": "gme_return",
    },
    "AMC": {
        "excess_return_file": "AMC-minute_price-excess-return.csv",
        "price_col":  "amc_price",
        "return_col": "amc_return",
    },
}

# ── Bar frequencies ────────────────────────────────────────────────────────────
FREQS = {
    "1min":  "1T",
    "5min":  "5T",
    "10min": "10T",
}

# ── Estimation window ──────────────────────────────────────────────────────────
ESTIMATION_WINDOW_DAYS = 120
FIRST_OUTAGE_START     = pd.Timestamp("2021-01-27 11:29:00")
ESTIMATION_END         = FIRST_OUTAGE_START - timedelta(days=1)

print(f"Estimation window end : {ESTIMATION_END}")
print(f"Estimation window start: {ESTIMATION_END - timedelta(days=ESTIMATION_WINDOW_DAYS)}")
print(f"Output directory      : {OUTPUT_DIR}")

---
## 2. Load 1-Minute Data

In [ ]:
def load_1min_stock(symbol):
    """Load the existing 1-min excess-return CSV for a stock."""
    path = DATA_DIR / STOCKS[symbol]["excess_return_file"]
    df = (
        pd.read_csv(path, parse_dates=["datetime"])
        .drop_duplicates("datetime")
        .sort_values("datetime")
        .set_index("datetime")
    )
    return df


def load_ff_factors():
    """Load minute-level Fama-French four factors (trading hours only)."""
    path = FF_DIR / "ff_factors_20201101_20210430_minute.csv"
    ff = (
        pd.read_csv(path, parse_dates=["datetime"])
        .drop_duplicates("datetime")
        .sort_values("datetime")
        .set_index("datetime")
    )
    return ff


gme_1min = load_1min_stock("GME")
amc_1min = load_1min_stock("AMC")
ff_1min  = load_ff_factors()

print(f"GME 1-min: {len(gme_1min):,} rows | {gme_1min.index.min()} → {gme_1min.index.max()}")
print(f"AMC 1-min: {len(amc_1min):,} rows | {amc_1min.index.min()} → {amc_1min.index.max()}")
print(f"FF  1-min: {len(ff_1min):,} rows  | {ff_1min.index.min()} → {ff_1min.index.max()}")

---
## 3. Resampling Functions

Resample 1-min bars to 5-min and 10-min:
- **Price**: last trade price in bar
- **Returns**: compound minute returns within bar → `(1+r₁)(1+r₂)... − 1`
- **FF factors**: sum (log-return-based factors are additive); `min_count=1` preserves NaN for fully after-hours bars

In [ ]:
def resample_stock(stock_1min, price_col, return_col, freq_alias):
    """
    Resample 1-min stock data to `freq_alias` (e.g. '5T').
    For 1T (1-min), returns the data as-is with renamed columns.
    """
    if freq_alias == "1T":
        out = stock_1min[[price_col, return_col, "spy_return"]].copy()
        out.columns = ["price", "return", "spy_return"]
        return out

    # Price: last trade in bar
    price = stock_1min[price_col].resample(freq_alias).last()

    # Returns: compound; mark empty bars (no trades) as NaN
    def compound(series, alias):
        bar = (1 + series.fillna(0)).resample(alias).prod() - 1
        count = series.resample(alias).count()
        bar[count == 0] = np.nan
        return bar

    ret = compound(stock_1min[return_col], freq_alias)
    spy = compound(stock_1min["spy_return"], freq_alias)

    return pd.DataFrame({"price": price, "return": ret, "spy_return": spy})


def resample_ff(ff_1min, freq_alias):
    """
    Resample 1-min FF factors to `freq_alias`.
    Uses sum (factors are log-return based); min_count=1 preserves NaN
    for bars where ALL minutes are after-hours (no factor data).
    """
    if freq_alias == "1T":
        return ff_1min[["MKT_RF", "SMB", "HML", "MOM"]].copy()
    return ff_1min[["MKT_RF", "SMB", "HML", "MOM"]].resample(freq_alias).sum(min_count=1)


print("Resample functions defined.")

---
## 4. Model Estimation Functions

Estimate CAPM, FF3, FF4 on the estimation window **at the target bar frequency**.
Inner join with FF factors restricts estimation to regular trading hours (09:30–16:00 ET).

In [ ]:
def estimate_models(bar_data, ff_bar, estimation_end,
                    window_days=ESTIMATION_WINDOW_DAYS):
    """
    Estimate CAPM, FF3, FF4 on bar-frequency data within the estimation window.

    Parameters
    ----------
    bar_data      : DataFrame with columns [price, return, spy_return]
    ff_bar        : DataFrame with columns [MKT_RF, SMB, HML, MOM]
    estimation_end: last timestamp of estimation window

    Returns
    -------
    (m_capm, m_ff3, m_ff4) : fitted OLS models
    """
    est_start = estimation_end - timedelta(days=window_days)

    # Stock returns in window
    w = bar_data.loc[est_start:estimation_end].copy()

    # Join FF factors — inner join keeps only bars with factor data (trading hours)
    w = w.join(ff_bar, how="inner")
    w = w.dropna(subset=["return", "spy_return", "MKT_RF", "SMB", "HML", "MOM"])

    y = w["return"]

    m_capm = sm.OLS(y, sm.add_constant(w[["spy_return"]])).fit()
    m_ff3  = sm.OLS(y, sm.add_constant(w[["MKT_RF", "SMB", "HML"]])).fit()
    m_ff4  = sm.OLS(y, sm.add_constant(w[["MKT_RF", "SMB", "HML", "MOM"]])).fit()

    return m_capm, m_ff3, m_ff4


def model_summary(symbol, freq_name, m_capm, m_ff3, m_ff4):
    """Print a compact parameter summary for all three models."""
    p4 = m_ff4.params
    print(f"  {symbol} @ {freq_name}  (n={int(m_capm.nobs):,} bars in estimation window)")
    print(f"    CAPM : α={m_capm.params.get('const',0):+.6f}  "
          f"β_mkt={m_capm.params.get('spy_return',0):.4f}  R²={m_capm.rsquared:.4f}  "
          f"σ={np.sqrt(m_capm.mse_resid):.6f}")
    print(f"    FF3  : α={m_ff3.params.get('const',0):+.6f}  "
          f"β_mkt={m_ff3.params.get('MKT_RF',0):.4f}  "
          f"β_smb={m_ff3.params.get('SMB',0):.4f}  "
          f"β_hml={m_ff3.params.get('HML',0):.4f}  R²={m_ff3.rsquared:.4f}")
    print(f"    FF4  : α={p4.get('const',0):+.6f}  "
          f"β_mkt={p4.get('MKT_RF',0):.4f}  "
          f"β_smb={p4.get('SMB',0):.4f}  "
          f"β_hml={p4.get('HML',0):.4f}  "
          f"β_mom={p4.get('MOM',0):.4f}  R²={m_ff4.rsquared:.4f}")


print("Model estimation functions defined.")

---
## 5. Abnormal Return Computation

Apply estimated model parameters to the **full time series** to produce AR and CAR columns.

- **CAPM AR**: available all hours (uses spy_return as market proxy)
- **FF3/FF4 AR**: NaN for after-hours bars where factors are unavailable

In [ ]:
def compute_ar_columns(bar_data, ff_bar, m_capm, m_ff3, m_ff4):
    """
    Compute expected returns, abnormal returns (AR), and cumulative AR (CAR)
    for all three models on the full bar time series.

    Returns a DataFrame with output columns only.
    """
    df = bar_data.join(ff_bar, how="left")  # left join: NaN for after-hours

    # ── CAPM (spy_return available all hours) ──────────────────────────────────
    p = m_capm.params
    df["expected_capm"] = p.get("const", 0) + p.get("spy_return", 0) * df["spy_return"]
    df["AR_capm"]       = df["return"] - df["expected_capm"]
    df["CAR_capm"]      = df["AR_capm"].cumsum()

    # ── FF3 (NaN after hours) ──────────────────────────────────────────────────
    p = m_ff3.params
    df["expected_ff3"] = (
        p.get("const",  0)
        + p.get("MKT_RF", 0) * df["MKT_RF"]
        + p.get("SMB",   0) * df["SMB"]
        + p.get("HML",   0) * df["HML"]
    )
    df["AR_ff3"]  = df["return"] - df["expected_ff3"]
    df["CAR_ff3"] = df["AR_ff3"].cumsum()

    # ── FF4 (NaN after hours) ──────────────────────────────────────────────────
    p = m_ff4.params
    df["expected_ff4"] = (
        p.get("const",  0)
        + p.get("MKT_RF", 0) * df["MKT_RF"]
        + p.get("SMB",   0) * df["SMB"]
        + p.get("HML",   0) * df["HML"]
        + p.get("MOM",   0) * df["MOM"]
    )
    df["AR_ff4"]  = df["return"] - df["expected_ff4"]
    df["CAR_ff4"] = df["AR_ff4"].cumsum()

    # ── Simple market-adjusted excess return ───────────────────────────────────
    df["excess_return_simple"] = df["return"] - df["spy_return"]

    output_cols = [
        "price", "return", "spy_return", "excess_return_simple",
        "MKT_RF", "SMB", "HML", "MOM",
        "expected_capm", "AR_capm", "CAR_capm",
        "expected_ff3",  "AR_ff3",  "CAR_ff3",
        "expected_ff4",  "AR_ff4",  "CAR_ff4",
    ]
    return df[[c for c in output_cols if c in df.columns]]


print("AR computation function defined.")

---
## 6. Main Loop — Estimate & Save All Frequencies

In [ ]:
# Store model parameters for summary comparison
model_params_store = {}

for freq_name, freq_alias in FREQS.items():
    print(f"\n{'='*65}")
    print(f"  Bar frequency: {freq_name}")
    print(f"{'='*65}")

    # Resample FF factors once per frequency
    ff_bar = resample_ff(ff_1min, freq_alias)

    for symbol, stock_1min in [("GME", gme_1min), ("AMC", amc_1min)]:
        info = STOCKS[symbol]

        # ── Resample stock data ───────────────────────────────────────────────
        bar_data = resample_stock(
            stock_1min, info["price_col"], info["return_col"], freq_alias
        )

        # ── Estimate models at this frequency ─────────────────────────────────
        m_capm, m_ff3, m_ff4 = estimate_models(bar_data, ff_bar, ESTIMATION_END)
        model_params_store[(symbol, freq_name)] = (m_capm, m_ff3, m_ff4)
        model_summary(symbol, freq_name, m_capm, m_ff3, m_ff4)

        # ── Compute AR/CAR columns ─────────────────────────────────────────────
        result = compute_ar_columns(bar_data, ff_bar, m_capm, m_ff3, m_ff4)

        # ── Save CSV ───────────────────────────────────────────────────────────
        out_fname = f"{symbol}-{freq_name}-multi-model-excess-return.csv"
        out_path  = OUTPUT_DIR / out_fname
        result.to_csv(out_path)
        print(f"    → Saved {out_fname}  ({len(result):,} bars)")

print("\nAll CSVs saved.")

---
## 7. Model Parameter Comparison Across Frequencies

Check that betas are stable across 1-min, 5-min, and 10-min frequencies.
R² should increase slightly at lower frequencies as microstructure noise diminishes.

In [ ]:
comparison_rows = []
for (symbol, freq_name), (m_capm, m_ff3, m_ff4) in model_params_store.items():
    for model_name, model in [("CAPM", m_capm), ("FF3", m_ff3), ("FF4", m_ff4)]:
        p = model.params
        comparison_rows.append({
            "Stock":    symbol,
            "Freq":     freq_name,
            "Model":    model_name,
            "alpha":    p.get("const",     0),
            "beta_mkt": p.get("spy_return", p.get("MKT_RF", 0)),
            "beta_smb": p.get("SMB",  np.nan),
            "beta_hml": p.get("HML",  np.nan),
            "beta_mom": p.get("MOM",  np.nan),
            "R2":       model.rsquared,
            "sigma":    np.sqrt(model.mse_resid),
            "n_obs":    int(model.nobs),
        })

comp_df = pd.DataFrame(comparison_rows)
comp_df.style.format({
    "alpha":    "{:.6f}",
    "beta_mkt": "{:.4f}",
    "beta_smb": "{:.4f}",
    "beta_hml": "{:.4f}",
    "beta_mom": "{:.4f}",
    "R2":       "{:.4f}",
    "sigma":    "{:.6f}",
}).set_caption("Model Parameters by Stock, Frequency, and Model")

---
## 8. Sample Output

In [ ]:
# Preview the 5-min GME CSV around the first outage window
preview_path = OUTPUT_DIR / "GME-5min-multi-model-excess-return.csv"
preview = pd.read_csv(preview_path, parse_dates=["datetime"]).set_index("datetime")

# Show rows around the first outage (Jan 27, 11:00–14:00)
mask = (preview.index >= "2021-01-27 11:00") & (preview.index <= "2021-01-27 14:00")
display_cols = ["price", "return", "excess_return_simple", "AR_capm", "AR_ff3", "AR_ff4",
                "CAR_capm", "CAR_ff3", "CAR_ff4"]
print("GME 5-min — Outage 1 window (Jan 27, 11:00–14:00):")
preview.loc[mask, display_cols].round(6)

In [ ]:
# Confirm all 6 output files exist
print("Output files:")
for freq_name in FREQS:
    for symbol in ["GME", "AMC"]:
        p = OUTPUT_DIR / f"{symbol}-{freq_name}-multi-model-excess-return.csv"
        size_mb = p.stat().st_size / 1e6 if p.exists() else 0
        status = f"{size_mb:.1f} MB" if p.exists() else "MISSING"
        print(f"  {'✓' if p.exists() else '✗'}  {p.name:<55}  {status}")